# L01 · 토큰·확률·자기회귀 LM

## Goal

**예상 시간:** 30분 · **경로:** full

- logit을 확률로 바꾼다
- next-token shape를 읽는다
- causal prefix를 확인한다

### 현재 위치: L00 → **L01** → L02

```text
Prompt/Data -> state source -> ... -> L01 -> ... -> fair evaluation
```

Alt text: The course map highlights L01 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L01"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L01', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

언어 모델은 매 위치에서 vocabulary 전체의 logit을 낸다. softmax는 이를 합이 1인 조건부 분포로 바꾸고, causal mask는 미래 토큰을 보지 못하게 한다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

자기회귀 모델은 `p(x_1:T)=Π_t p(x_t | x_<t)`로 sequence 확률을 분해한다. 구현에서 logits shape는 `[batch, time, vocabulary]`이고, 위치 `t`의 logits는 target `t+1`과 비교된다. 이 한 칸 shift를 빠뜨리면 미래 token을 그대로 맞히는 잘못된 학습이 된다.

padding mask는 존재하지 않는 위치를, response mask는 prompt를 loss에서 제외한다. causal mask는 attention 자체가 미래를 보지 못하게 한다. 세 mask는 목적이 다르므로 하나로 대체할 수 없다.

### 실제 구현: 왜 이렇게 만들었나

구현 순서는 `token IDs → token/position embedding → causal Transformer → layer norm → vocabulary logits`다. `TinyCausalLM.forward`는 입력 rank와 최대 길이를 검사하고 causal/padding mask를 분리한다. `generate`는 greedy(`temperature=0`)와 stochastic sampling을 명시적으로 나눈다.

실제 코드: [`tiny_transformer.py`](../../src/opd_study/models/tiny_transformer.py), [`tokenizer.py`](../../src/opd_study/data/tokenizer.py).

In [2]:
import inspect
from opd_study.models import TinyCausalLM

objects_to_show = (TinyCausalLM.forward, TinyCausalLM.generate,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.models.tiny_transformer.TinyCausalLM.forward
    def forward(self, token_ids: Tensor, attention_mask: Tensor | None = None) -> Tensor:
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape [batch, sequence]")
        _, sequence_length = token_ids.shape
        if sequence_length < 1 or sequence_length > self.config.max_sequence_length:
            raise ValueError(
                f"sequence length must be within [1, {self.config.max_sequence_length}], "
                f"got {sequence_length}"
            )
        if attention_mask is None:
            attention_mask = torch.ones_like(token_ids, dtype=torch.bool)
        if attention_mask.shape != token_ids.shape:
            raise ValueError("attention_mask must have the same shape as token_ids")
        positions = torch.arange(sequence_length, device=token_ids.device)
        hidden = self.token_embedding(token_ids) * math.sqrt(self.config.hidden_size)
        hidden = hidden + self.

### 다른 선택지는 없나?

실제 LLM은 subword tokenizer, RoPE, tied embeddings, RMSNorm, FlashAttention 등을 쓸 수 있다. 이 강좌는 character tokenizer와 표준 Transformer를 택해 token 경계와 mask를 눈으로 검사하게 한다. 이 선택은 교육용이며 Qwen backend가 같은 구조라는 뜻이 아니다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L01의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
logits = torch.tensor([2.0, 1.0, 0.0])
probabilities = torch.softmax(logits, dim=-1)
manual = logits.exp() / logits.exp().sum()
print("probabilities:", probabilities.tolist())
print("sum:", probabilities.sum().item(), "manual match:", torch.allclose(probabilities, manual))

probabilities: [0.6652409434318542, 0.2447284758090973, 0.09003057330846786]
sum: 1.0 manual match: True


In [4]:
from opd_study.models import TinyCausalLM, TinyTransformerConfig

config = TinyTransformerConfig(vocab_size=12, max_sequence_length=16,
                               number_of_layers=1, hidden_size=16,
                               number_of_heads=4, feed_forward_size=32)
model = TinyCausalLM(config).eval()
prefix_a = torch.tensor([[1, 4, 5, 6]])
prefix_b = prefix_a.clone(); prefix_b[0, -1] = 7
with torch.no_grad():
    logits_a, logits_b = model(prefix_a), model(prefix_b)
print("shape:", tuple(logits_a.shape),
      "past unchanged:", torch.allclose(logits_a[:, :-1], logits_b[:, :-1]))

shape: (1, 4, 12) past unchanged: True


## Checks

In [5]:
assert probabilities.shape == (3,)
assert torch.isclose(probabilities.sum(), torch.tensor(1.0))
assert torch.allclose(logits_a[:, :-1], logits_b[:, :-1])
print("check passed: normalized next-token probabilities and causal prefix")

check passed: normalized next-token probabilities and causal prefix


**연습 (5분):** `prefix_b`의 마지막 token이 아니라 첫 token을 바꾸면 어느 위치 logits부터 달라져야 하는지 예측하고 assertion을 작성하라.

<details><summary>확인 기준</summary>바뀐 첫 token 이후 위치는 달라질 수 있지만 그보다 과거 위치는 없다. 미래 token이 과거 logits를 바꾸지 않는 방향을 검사한다.</details>

## 내가 자주 틀리는 것

### M1 — logits를 이미 확률이라고 생각하기

- 틀린 형태: logits 합이 1이라고 가정한다.
- 왜 틀렸나: logits는 정규화되지 않은 실수 score다.
- 고친 형태: vocabulary 축 softmax와 합을 확인한다.
- 관련 검사: `test_future_token_does_not_change_past_logits`

### M2 — causal mask와 response mask를 합치기

- 틀린 형태: 미래 차단 mask로 prompt loss까지 제거됐다고 생각한다.
- 왜 틀렸나: attention 접근성과 objective 포함 여부는 다르다.
- 고친 형태: causal/attention/response mask를 따로 감사한다.
- 관련 검사: `test_sft_counts_only_response_targets`

## 60초 요약

1. logit을 확률로 바꾼다
2. next-token shape를 읽는다
3. causal prefix를 확인한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)